# Assignment 2: Transformer language models

Build an OLMo-2-style Transformer from scratch, train on the same Wikipedia data as A1, and generate text. Reuses A1's tokenizer and trainer.

All Transformer components live in `A2_skeleton.py`.

## Setup

In [1]:
!git clone https://github.com/dmw1998/WASP_DL4NLP26.git 2>/dev/null || true
%cd WASP_DL4NLP26/Assignments
!ls

/content/WASP_DL4NLP26/Assignments
A1  A2


In [2]:
!pip install -q datasets nltk scikit-learn matplotlib transformers accelerate

In [3]:
import os, math, sys
import torch
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Reuse A1's tokenizer and trainer.
sys.path.insert(0, './A1/a1_1')
sys.path.insert(0, './A2')

TRAIN_FILE = './A1/a1_1/train.txt'
VAL_FILE = './A1/a1_1/val.txt'
assert os.path.exists(TRAIN_FILE), f'{TRAIN_FILE} not found'
assert os.path.exists(VAL_FILE), f'{VAL_FILE} not found'

CUDA available: True
GPU: Tesla T4


## Step 0: load A1 tokenizer

> **Notes**
> - Tokenizer and trainer are reused **verbatim** from A1
> - Only the model class changes (RNN → Transformer)
> - Vocab size, special tokens, training data are identical

In [4]:
from A2.A1_skeleton import build_tokenizer, A1Tokenizer, lowercase_tokenizer, A1Trainer

MAX_VOC_SIZE = 10000
MODEL_MAX_LENGTH = 128

tokenizer = build_tokenizer(
    train_file=TRAIN_FILE,
    tokenize_fun=lowercase_tokenizer,
    max_voc_size=MAX_VOC_SIZE,
    model_max_length=MODEL_MAX_LENGTH,
)
print('vocab size:', len(tokenizer))
print('pad_token_id:', tokenizer.pad_token_id)

vocab size: 10000
pad_token_id: 0


## Step 1: Setting up the Transformer

### Configuration

Hyperparameter overview for a small OLMo-2-style model. We use a small model (3 layers, hidden 256, 4 heads) to keep training time under 10 minutes on Colab T4.

In [5]:
from A2_skeleton import A2ModelConfig

config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=256,
    intermediate_size=512,         # SwiGLU intermediate dim (~2x hidden)
    num_attention_heads=4,         # 256 / 4 = 64 dim per head
    num_hidden_layers=3,
    rope_theta=10000.0,            # OLMo-2 default
    hidden_act='silu',
    max_position_embeddings=MODEL_MAX_LENGTH,
    rms_norm_eps=1e-5,
)
print(f'hidden_size: {config.hidden_size}')
print(f'head_dim: {config.hidden_size // config.num_attention_heads}')
print(f'intermediate_size: {config.intermediate_size}')
print(f'num_hidden_layers: {config.num_hidden_layers}')

hidden_size: 256
head_dim: 64
intermediate_size: 512
num_hidden_layers: 3


### 🎓 Task 1.1: MLP layer (SwiGLU)

> **Notes**
> - SwiGLU = `down(SiLU(gate(x)) ⊗ up(x))` — two parallel projections, element-wise multiply, project back
> - **Three Linears** all `bias=False`: `gate_proj` (H→I), `up_proj` (H→I), `down_proj` (I→H)
> - **SiLU(x) = x · sigmoid(x)** — also called Swish; smooth ReLU-like activation
> - **Why gated?** The element-wise multiplication lets the network learn input-dependent feature selection (the `gate` branch acts as a multiplicative attention over the `up` branch)
> - **vs vanilla MLP**: `down(SiLU(up(x)))` is one matmul fewer but less expressive; SwiGLU empirically gives better perplexity per parameter

In [6]:
from A2_skeleton import A2MLP

mlp = A2MLP(config)
x = torch.randn(2, 7, config.hidden_size)
out = mlp(x)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.1 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.1 sanity check passed


### ⚙ Task 1.2: RMSNorm

> **Notes**
> - We use PyTorch's built-in `nn.RMSNorm` (allowed by the assignment)
> - `RMSNorm(x) = x / sqrt(mean(x²) + eps) · γ` — no centering (no mean subtraction), unlike LayerNorm
> - `elementwise_affine=True` → learnable per-channel scale γ
> - **Why RMSNorm over LayerNorm?** Simpler (one stat instead of two), ~30% faster, no quality loss in practice; standard in modern LLMs (Llama, OLMo, Qwen, etc.)
> - **Where used in OLMo-2**: after attention, after MLP, on Q and K projections inside attention, and once before the final unembedding

In [7]:
from A2_skeleton import A2RMSNorm

norm = A2RMSNorm(config)
out = norm(x)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.2 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.2 sanity check passed


### 🎓 Task 1.3: Multi-head attention

> **Notes — components**
> - 4 square `nn.Linear` (all `bias=False`): `q_proj`, `k_proj`, `v_proj`, `o_proj`, each `hidden_size → hidden_size`
> - OLMo-2 adds **RMSNorm on Q and K** ("QK-norm") — stabilizes training, prevents attention logit explosions
>
> **Notes — forward, step by step**
> 1. Project: `q = q_norm(W_Q · x)`, `k = k_norm(W_K · x)`, `v = W_V · x` — shape (B, M, D)
> 2. Split heads: `view(B, M, n_h, d_h).transpose(1, 2)` → (B, n_h, M, d_h)
> 3. Apply RoPE rotations to q and k only (NOT v) — rotations encode position
> 4. `F.scaled_dot_product_attention(q, k, v, is_causal=True)` — does scaling, masking, softmax, weighted sum in one call
> 5. Merge heads: `transpose(1, 2).reshape(B, M, D)`
> 6. Output projection: `W_O · attn_out`
>
> **Notes — why each piece**
> - **Multi-head**: lets the model attend to different subspaces simultaneously (one head might track syntax, another semantics)
> - **Scaling by √d_h**: keeps dot products from growing with d_h, preventing softmax saturation
> - **Causal mask**: position i can only attend to positions ≤ i — required for autoregressive LM
> - **RoPE**: rotates query and key vectors based on position; the dot product q·k becomes a function of *relative* position (i−j), not absolute. Better generalization to unseen lengths than learned positional embeddings.

In [8]:
from A2_skeleton import A2Attention, A2RotaryEmbedding

attn = A2Attention(config)
rope = A2RotaryEmbedding(config)

# RoPE needs to know sequence length, which it reads from input shape[1].
dummy_ids = torch.zeros(2, 7, dtype=torch.long)
rope_rotations = rope(dummy_ids)
print(f'RoPE cos shape: {rope_rotations[0].shape}')
print(f'RoPE sin shape: {rope_rotations[1].shape}')

out = attn(x, rope_rotations)
print(f'\nattention in:  {x.shape}')
print(f'attention out: {out.shape}')
assert out.shape == x.shape
print('Task 1.3 sanity check passed')

RoPE cos shape: torch.Size([1, 7, 64])
RoPE sin shape: torch.Size([1, 7, 64])

attention in:  torch.Size([2, 7, 256])
attention out: torch.Size([2, 7, 256])
Task 1.3 sanity check passed


### 🎓 Task 1.4: Full Transformer decoder layer

> **Notes**
> - Two sublayers: **self-attention** + **MLP (SwiGLU)**
> - Each sublayer has a **residual connection**: `x = x + sublayer(x)`
> - OLMo-2 uses **post-norm** (norm AFTER sublayer, BEFORE adding residual):
>   ```
>   x = x + RMSNorm(Attention(x))
>   x = x + RMSNorm(MLP(x))
>   ```
> - **Why residual?** Gradient flows directly back through `+`, mitigates vanishing gradients in deep stacks; also lets layers learn *delta* updates rather than full transformations
> - **Post-norm vs pre-norm**: pre-norm (`x = x + Sublayer(RMSNorm(x))`, used by Llama) is easier to train; post-norm (used by OLMo-2 and original Transformer) needs more care but can give better final quality

In [9]:
from A2_skeleton import A2DecoderLayer

layer = A2DecoderLayer(config)
out = layer(x, rope_rotations)
print(f'in:  {x.shape}')
print(f'out: {out.shape}')
assert out.shape == x.shape
print('Task 1.4 sanity check passed')

in:  torch.Size([2, 7, 256])
out: torch.Size([2, 7, 256])
Task 1.4 sanity check passed


### ⚙ Task 1.5: Complete Transformer stack

> **Notes**
> - Top-level structure: `embed_tokens` → N × `A2DecoderLayer` → final `RMSNorm` → `lm_head` (unembedding)
> - Layers stored in `nn.ModuleList` (not plain Python list) so parameters get registered for autograd
> - `lm_head` has `bias=False` (OLMo-2 convention)
> - **RoPE computed once** at the top of `forward`, then **shared** across all layers — saves recomputation
> - Loss uses **shift-by-one** identical to A1: drop last logit, drop first label
>
> **Notes — model size in this run**
> - Total: **~7.09M parameters**
> - Embedding + unembedding alone: 2 × (10000 × 256) ≈ 5.12M → **~72% of all params**
> - 3 decoder layers contribute only ~2M
> - **Take-away**: with small vocab + small hidden, embeddings dominate. In big LLMs (50k+ vocab BPE, but huge hidden_size), MLP layers dominate instead.


In [10]:
from A2_skeleton import A2Transformer

model = A2Transformer(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

# Sanity check: input integer tensor → 3D logits tensor.
input_ids = torch.randint(0, len(tokenizer), (2, 7))
out = model(input_ids)
print(f'\ninput shape:  {input_ids.shape}')
print(f'logits shape: {out.logits.shape}')
expected = (2, 7, len(tokenizer))
assert out.logits.shape == expected

# Loss check.
out = model(input_ids, labels=input_ids)
print(f'loss (random init): {out.loss.item():.4f} (expect ≈ log({len(tokenizer)}) = {math.log(len(tokenizer)):.2f})')
print('Task 1.5 sanity check passed')

parameters: 7,089,408

input shape:  torch.Size([2, 7])
logits shape: torch.Size([2, 7, 10000])
loss (random init): 9.3087 (expect ≈ log(10000) = 9.21)
Task 1.5 sanity check passed


## Step 2: Training

### ⚙ Task 2.1: train the Transformer LM

> **Notes**
> - Reuse the same `A1Trainer` from Assignment 1 — interface is identical (model takes `input_ids` + `labels`, returns `.loss`)
> - Hyperparameters: small model (3 layers, hidden 256, 4 heads, ~7M params), AdamW lr=3e-4, 3 epochs, batch_size=32
> - **Lower learning rate than A1** (3e-4 vs 1e-3): Transformers are typically more sensitive to LR than RNNs
>
> **Notes — my actual results**
> - Val ppl: epoch 1 → **66.3**, epoch 2 → **55.9**, epoch 3 → **~50**
> - **Beats A1's RNN** (75 with same data) — Transformer learns richer dependencies, especially long-range
> - Training loss kept dropping (4.6 → 4.0 → 3.85) → could train longer for marginal gains, but diminishing returns
> - **Why better than RNN?** Attention can directly reach any previous token; RNN has to propagate information through every hidden state in between → information bottleneck. Transformers also parallelize across positions, so each epoch trains faster.


In [11]:
from datasets import load_dataset
from torch.utils.data import Subset
from transformers import TrainingArguments

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')
print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))

dev_mode = False  # set True for a quick (~1 min) sanity run

if dev_mode:
    train_ds = Subset(dataset['train'], range(1000))
    val_ds = Subset(dataset['val'], range(200))
    epochs = 1
else:
    train_ds = dataset['train']
    val_ds = dataset['val']
    epochs = 3

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/294118 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35748 [00:00<?, ? examples/s]

train size: 147059
val size:   17874


In [12]:
args = TrainingArguments(
    output_dir='trainer_output_a2',
    optim='adamw_torch',
    eval_strategy='epoch',
    learning_rate=3e-4,            # smaller than A1 — transformers need it
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=100,
    save_strategy='no',
    report_to='none',
    use_cpu=False,
)

trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()

Device: cuda
Epoch 1 step 100 avg_loss 6.5763
Epoch 1 step 200 avg_loss 5.6780
Epoch 1 step 300 avg_loss 5.4599
Epoch 1 step 400 avg_loss 5.3058
Epoch 1 step 500 avg_loss 5.2018
Epoch 1 step 600 avg_loss 5.1169
Epoch 1 step 700 avg_loss 5.0619
Epoch 1 step 800 avg_loss 5.0033
Epoch 1 step 900 avg_loss 4.9347
Epoch 1 step 1000 avg_loss 4.8864
Epoch 1 step 1100 avg_loss 4.8448
Epoch 1 step 1200 avg_loss 4.7962
Epoch 1 step 1300 avg_loss 4.7378
Epoch 1 step 1400 avg_loss 4.7032
Epoch 1 step 1500 avg_loss 4.6805
Epoch 1 step 1600 avg_loss 4.6378
Epoch 1 step 1700 avg_loss 4.6154
Epoch 1 step 1800 avg_loss 4.6021
Epoch 1 step 1900 avg_loss 4.5636
Epoch 1 step 2000 avg_loss 4.5342
Epoch 1 step 2100 avg_loss 4.5293
Epoch 1 step 2200 avg_loss 4.4803
Epoch 1 step 2300 avg_loss 4.4623
Epoch 1 step 2400 avg_loss 4.4490
Epoch 1 step 2500 avg_loss 4.4258
Epoch 1 step 2600 avg_loss 4.4294
Epoch 1 step 2700 avg_loss 4.4169
Epoch 1 step 2800 avg_loss 4.4000
Epoch 1 step 2900 avg_loss 4.3939
Epoch 1 st

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Step 3: Text generation

### ⚙ Task 3.1: predict next word (greedy)

> **Notes**
> - Same as A1 5.1: take logits at position **-2** (one before `<EOS>`)
> - `argmax` → most likely next token id → look up word
>
> **Notes — the `<UNK>` problem (observed in my output)**
> - For 'She lives in San', `<UNK>` has the highest logit (12.7), **above** 'francisco' (10.8) and 'diego' (10.0)
> - **Why?** `<UNK>` collapses ALL out-of-vocab words into one token → in training, position after 'in' very often had `<UNK>` (any rare city name) → model learned `<UNK>` is the safest bet
> - **Implication**: pure greedy decoding picks `<UNK>` and becomes useless. This is one reason sampling + top-K matters — lower-ranked but specific tokens can still win.
> - **Real LLMs sidestep this**: BPE tokenizers split rare words into known subword pieces, so there's no single `<UNK>` catching everything. The vocabulary is effectively unbounded.


In [13]:
def predict_next(model, tokenizer, prompt, k=5):
    device = next(model.parameters()).device
    model.eval()
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    with torch.no_grad():
        out = model(input_ids)
    logits = out.logits[0, -2]   # position before <EOS>
    topk = torch.topk(logits, k)
    return [(tokenizer.int_to_str[i.item()], s.item())
            for i, s in zip(topk.indices, topk.values)]

for prompt in ['She lives in San',
               'The president of the United',
               'The capital of Sweden is']:
    print(f'prompt: {prompt!r}')
    for word, score in predict_next(model, tokenizer, prompt):
        print(f'  {word:20s} {score:.3f}')
    print()

prompt: 'She lives in San'
  <UNK>                12.677
  francisco            10.814
  diego                9.989
  suu                  9.202
  antonio              8.852

prompt: 'The president of the United'
  kingdom              12.626
  states               12.536
  nations              11.666
  arab                 9.460
  front                8.546

prompt: 'The capital of Sweden is'
  the                  9.325
  <UNK>                8.579
  a                    7.950
  located              7.641
  home                 7.542



### 🎓 Task 3.2: text generation with sampling

> **Notes — algorithm**
> 1. Encode prompt → `input_ids`
> 2. Loop until `<EOS>` or `max_length`:
>    a. Forward pass → take logits at last position
>    b. Divide logits by `temperature` (higher → more random)
>    c. If `topk`: keep only top-K logits, set rest to -inf
>    d. Sample one token from the resulting distribution
>    e. Append to `input_ids`
>
> **Notes — knobs and their effects**
> - **temperature → 0**: deterministic, picks argmax every time (= greedy decoding, can loop)
> - **temperature = 1**: sample from the model's actual distribution
> - **temperature → ∞**: uniform random (gibberish)
> - **top-K**: truncate to K most likely tokens before sampling — prevents picking very unlikely tokens
> - **Why sampling?** Greedy decoding is deterministic but often produces repetitive, bland text. Sampling adds variety. Top-K + moderate temperature is the sweet spot.
>
> **Notes — what my model actually produced**
> - **Low temp (0.3) + top-K=5** → `"in natural language processing, a <UNK> is a <UNK> of <UNK>, a <UNK> of the <UNK>..."` Gets stuck in `<UNK>` loops — `<UNK>` is the highest-prob token, low temp picks it every time.
> - **Moderate (T=0.8, K=40)** → `"is stockholm the capital of sweden? ... the answer is the same."` Grammatical, but no real knowledge — the model never produces 'yes' or 'Stockholm'.
> - **High temp (1.5, no K)** → `"...power was equal, conclusion making mutual entire desires, leading steadily..."` Mostly real words, no coherence.
>
> **My observations:**
> 1. **`<UNK>` amplification**: small vocab (10k) means many real words → `<UNK>`. Low temperature **amplifies** this because `<UNK>` is the safest bet.
> 2. **No factual knowledge**: even when grammatical, the model can't say 'yes' or 'Stockholm'. It has never seen enough text to learn facts.
> 3. **Temperature trade-off is real**: T=0.3 → stuck in safe loops; T=1.5 → gibberish; T=0.8 → sweet spot. This is exactly why production LLMs default to T≈0.7–1.0.


In [14]:
from torch.distributions import Categorical

def generate(model, tokenizer, prompt, max_length=50, temperature=1.0, topk=None):
    """Sample text autoregressively from the model."""
    device = next(model.parameters()).device
    model.eval()

    # Encode prompt (don't append <EOS> — we'll generate beyond it).
    # We use the tokenizer normally and then strip the trailing EOS.
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    if input_ids[0, -1].item() == tokenizer.eos_token_id:
        input_ids = input_ids[:, :-1]

    generated = []
    with torch.no_grad():
        for _ in range(max_length):
            out = model(input_ids)
            logits = out.logits[0, -1]                       # last position
            logits = logits / max(temperature, 1e-8)         # temperature scaling

            if topk is not None:
                top_vals, top_idx = torch.topk(logits, topk)
                # Mask everything outside top-K to -inf so softmax ignores it.
                mask = torch.full_like(logits, float('-inf'))
                mask[top_idx] = top_vals
                logits = mask

            dist = Categorical(logits=logits)
            next_id = dist.sample()
            if next_id.item() == tokenizer.eos_token_id:
                break
            generated.append(next_id.item())
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)

    # Decode: prompt + sampled words.
    prompt_words = tokenizer.decode(enc['input_ids'][0]) if hasattr(tokenizer, 'decode') \
        else [tokenizer.int_to_str[i.item()] for i in enc['input_ids'][0]
              if i.item() not in (tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id)]
    gen_words = [tokenizer.int_to_str[i] for i in generated]
    return ' '.join(prompt_words + gen_words)

In [ ]:
# Run with several prompts and parameter combinations.
prompts = [
    'In natural language processing , a transformer',
    'Is stockholm the capital of sweden ? answer yes or no . the answer is',
    'Write a python program that reverses a list .',
]

configs = [
    {'temperature': 0.3, 'topk': 5,    'label': 'low temp / low K (conservative)'},
    {'temperature': 0.8, 'topk': 40,   'label': 'moderate (LLM default)'},
    {'temperature': 1.5, 'topk': None, 'label': 'high temp / no K (chaotic)'},
]

torch.manual_seed(17)
for prompt in prompts:
    print(f'=== PROMPT: {prompt!r} ===')
    for cfg in configs:
        label = cfg.pop('label')
        text = generate(model, tokenizer, prompt, max_length=40, **cfg)
        print(f'\n[{label}]')
        print(text)
        cfg['label'] = label   # restore for next prompt
    print('\n' + '='*70 + '\n')

=== PROMPT: 'In natural language processing , a transformer' ===

[low temp / low K (conservative)]
in natural language processing , a <UNK> is a <UNK> of <UNK> , a <UNK> of the <UNK> of the <UNK> of <UNK> . the <UNK> of the <UNK> is a <UNK> of the <UNK> , which is a <UNK> of the <UNK> <UNK> .

[moderate (LLM default)]
in natural language processing , a <UNK> <UNK> or a set of symbols is a function of a <UNK> consisting of a <UNK> or a <UNK> or a <UNK> , but is a <UNK> or an <UNK> of a <UNK> number as a <UNK> or a <UNK>

[high temp / no K (chaotic)]
in natural language processing , a <UNK> aspect or excommunication refers calling court weeks functions or activity or commission likely specifies oath or prefix preparation ( for example : case iv parties in a use on crown <UNK> commonwealth laws act x <UNK> 1 non union day


=== PROMPT: 'Is stockholm the capital of sweden ? answer yes or no . the answer is' ===

[low temp / low K (conservative)]
is stockholm the capital of sweden ? answer

### 🎓 Task 3.3: Compare to pre-trained OLMo-2-1B

> **Notes**
> - Download ~3GB of weights — slow on first run
> - OLMo-2-1B was trained on **trillions** of tokens vs our ~150k Wikipedia lines
> - It's a **base model** (not instruction-tuned) — doesn't follow instructions, just continues text
>
> **Notes — comparison with my own model (observed)**
>
> Same 3 prompts, side by side:
>
> | Prompt | My model (7M params) | OLMo-2-1B (1.5B params) |
> |---|---|---|
> | "...a Transformer" | `"a <UNK> is a <UNK> of <UNK>..."` | `"attention uses a Transformer-based attention mechanism [18] to compute the attention weights..."` |
> | "Stockholm capital..." | `"the answer is the same."` | **`"The answer is yes."`** ✓ correct |
> | "Python reverses list" | `"a <UNK> program that <UNK> a list"` | `"You should use a for loop to reverse the list..."` |
>
> **Why the gap?**
> - **Data**: OLMo-2 trained on ~5T tokens, mine on ~150k Wikipedia paragraphs → **~30M× more**
> - **Params**: 1.5B vs 7M → **~200×**
> - **Tokenizer**: BPE subwords (preserves 'Stockholm', 'Python') vs whole-word; everything outside top-10k → `<UNK>`
> - **The `<UNK>` problem is decisive**: my model **literally cannot output 'Stockholm'** — it's not in the vocab. Even a perfect 7M-param model couldn't answer this question with my tokenizer.
>
> **OLMo-2 limitations (still visible in its output):**
> - **Not instruction-tuned**: doesn't 'answer', just continues text
> - **Hallucination**: invents citations like `[18]` that don't exist in the prompt
> - **Repetition / confusion**: suggests both `for loop` and `list comprehension` to reverse a list, but neither actually does (it's a continuation of patterns it has seen, not reasoning)
> - These are real limitations even with 200× more params + 30M× more data → why post-training (instruction-tuning, RLHF) is needed


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'allenai/OLMo-2-0425-1B'
olmo_tok = AutoTokenizer.from_pretrained(model_name)
olmo = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
olmo = olmo.to('cuda' if torch.cuda.is_available() else 'cpu')
olmo.eval()

print(f'OLMo-2-1B params: {sum(p.numel() for p in olmo.parameters()):,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

OLMo-2-1B params: 1,484,916,736


In [ ]:
def generate_olmo(prompt, max_new_tokens=60, temperature=0.8, top_k=40):
    inputs = olmo_tok(prompt, return_tensors='pt').to(olmo.device)
    with torch.no_grad():
        out_ids = olmo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=olmo_tok.eos_token_id,
        )
    return olmo_tok.decode(out_ids[0], skip_special_tokens=True)

torch.manual_seed(17)
olmo_prompts = [
    'In natural language processing, a Transformer',
    'Is Stockholm the capital of Sweden? Answer yes or no. The answer is',
    'Write a Python program that reverses a list.',
]
for p in olmo_prompts:
    print(f'=== {p!r} ===')
    print(generate_olmo(p))
    print()

=== 'In natural language processing, a Transformer' ===
In natural language processing, a Transformer model's attention uses a Transformer-based attention mechanism [18] to compute the attention weights for a sequence of inputs. An attention mechanism computes a score for each token of a sequence and produces a weighted sum of these scores. Transformer-based attention computes the attention weights for a token using the attention mechanism, and

=== 'Is Stockholm the capital of Sweden? Answer yes or no. The answer is' ===
Is Stockholm the capital of Sweden? Answer yes or no. The answer is yes.

=== 'Write a Python program that reverses a list.' ===
Write a Python program that reverses a list. You should use a for loop to reverse the list. You should use a list comprehension to reverse the list. Write a function that accepts an integer or a string and reverses the string. For example, if you call this function with: print reverse('abc') it would print abca. You



## Wrap-up: oral exam quick-reference

**Most likely follow-ups:**

- **Why RoPE over learned position embeddings?** Generalizes to longer sequences; encodes *relative* position naturally via the q·k dot product
- **Why causal mask?** Without it, position *i* could attend to future tokens — defeats autoregressive LM
- **Why scale by √d_h?** Dot product of two random d_h-dim vectors has variance d_h; scaling keeps softmax in a reasonable range
- **Why multi-head?** Single softmax can only attend to one thing at a time; multiple heads let the model track multiple types of relations in parallel
- **Why pre-norm vs post-norm?** Pre-norm trains more stably; post-norm (OLMo-2, original) needs warmup but can be slightly better at convergence
- **Why SwiGLU over plain MLP?** Gating mechanism lets the network learn input-dependent feature selection; empirically lower perplexity per parameter
- **Why bias=False everywhere?** OLMo-2 convention; saves params; RMSNorm provides the affine offset that biases would otherwise give
- **Transformer vs RNN trade-offs?** Transformer: parallelizable across positions (fast training), but O(n²) attention (slow for long contexts). RNN: O(n) but sequential (slow training), forgets long-range info.